# CO_multi_quantile_panel

Three pooled-analysis tables for conformal VaR evaluation.

- **Table 5:** Multi-quantile evaluation for TSFMs at four quantile levels.
- **Table 6:** Panel-pooled backtest at α = 0.01 for all 9 models.
- **Table 7:** Panel-pooled backtest by asset class (α = 0.01).

**Outputs:** `tab_multiquantile.tex`, `tab_panel_pooled.tex`, `tab_panel_by_class.tex`  
**Paper:** Pele, Bolovaneanu, Ginavar, Lessmann, Härdle (2026)  
**Q** [Conformal_Oracle](https://github.com/QuantLet/Conformal_Oracle/)

In [ ]:
"""
CO_multi_quantile_panel — Multi-quantile evaluation for TSFMs.
Produces tab_multiquantile.tex  (Table 5 in the paper).
"""

import pandas as pd
import numpy as np
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP

# ── Paths ──────────────────────────────────────────────────────────
DATA_DIR = Path(__file__).resolve().parent.parent / 'cfp_ijf_data'
RES_DIR  = DATA_DIR / 'paper_outputs' / 'tables'
OUT_DIR  = Path(__file__).resolve().parent

TSFM_ORDER = ['Chronos-Small', 'Chronos-Mini', 'TimesFM-2.5',
              'Moirai-2.0', 'Lag-Llama']
ALPHAS = [0.01, 0.025, 0.05, 0.10]

# ── Load ───────────────────────────────────────────────────────────
df = pd.read_csv(RES_DIR / 'all_results.csv')
df = df[df['model'].isin(TSFM_ORDER)].copy()
print(f'Loaded {len(df)} TSFM rows '
      f'({df["model"].nunique()} models, '
      f'{df["symbol"].nunique()} assets, '
      f'{df["alpha"].nunique()} alphas)')

# ── Compute 5 × 4 × 2 cells ──────────────────────────────────────
rows = []
for model in TSFM_ORDER:
    for alpha in ALPHAS:
        sub = df[(df['model'] == model) & (df['alpha'] == alpha)]
        pi_hat = sub['pihat_cp'].mean()
        rej = int((sub['p_kup_cp'] < 0.05).sum())
        n_assets = len(sub)
        rows.append({
            'model': model, 'alpha': alpha,
            'pi_hat': pi_hat, 'rej': rej, 'n_assets': n_assets,
        })

result = pd.DataFrame(rows)
print('\nMulti-quantile table:')
for _, r in result.iterrows():
    print(f'  {r["model"]:16s}  α={r["alpha"]:.3f}  '
          f'π̂={r["pi_hat"]:.3f}  Rej={r["rej"]:2d}/{r["n_assets"]}')

# ── Save CSV ──────────────────────────────────────────────────────
result.to_csv(OUT_DIR / 'tab_multiquantile.csv', index=False)

# ── Save LaTeX ────────────────────────────────────────────────────
lines = [
    r'\begin{tabular}{@{}l rr rr rr rr@{}}',
    r'\toprule',
    r'& \multicolumn{2}{c}{$\alpha=0.01$}',
    r'& \multicolumn{2}{c}{$\alpha=0.025$}',
    r'& \multicolumn{2}{c}{$\alpha=0.05$}',
    r'& \multicolumn{2}{c}{$\alpha=0.10$} \\',
    r'\cmidrule(lr){2-3}\cmidrule(lr){4-5}',
    r'\cmidrule(lr){6-7}\cmidrule(l){8-9}',
    r'Model & $\hat\pi$ & Rej.',
    r'& $\hat\pi$ & Rej.',
    r'& $\hat\pi$ & Rej.',
    r'& $\hat\pi$ & Rej. \\',
    r'\midrule',
]

for model in TSFM_ORDER:
    label = model
    if model == 'TimesFM-2.5':
        label = 'TimesFM 2.5'
    elif model == 'Moirai-2.0':
        label = 'Moirai 2.0'

    cells = []
    for alpha in ALPHAS:
        r = result[(result['model'] == model) & (result['alpha'] == alpha)].iloc[0]
        pi_dec = Decimal(str(r['pi_hat'])).quantize(
            Decimal('0.001'), rounding=ROUND_HALF_UP)
        pi_str = f'.{int(pi_dec * 1000):03d}'
        rej_str = f'{r["rej"]:d}/{r["n_assets"]}'
        cells.append(f'{pi_str} & {rej_str}')

    line = f'{label} & ' + '\n& '.join(cells) + r' \\'
    lines.append(line)

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')

tex = '\n'.join(lines) + '\n'
tex_path = OUT_DIR / 'tab_multiquantile.tex'
tex_path.write_text(tex)
print(f'\nSaved {tex_path.name}')
print(tex)

## Table 6 — Panel-pooled backtest (α = 0.01)

In [ ]:
"""
CO_multi_quantile_panel — Panel-pooled backtest at alpha=0.01.
Produces tab_panel_pooled.tex  (Table 6 in the paper).

Every cell is computed from reproducible inputs:
  - Violation indicators: forecast parquets + returns + qV
  - N_panel, violations, pi_pooled: from violation counts
  - HAC SE: Driscoll-Kraay panel estimator
      Kernel:    Bartlett (Newey-West 1987)
      Bandwidth: Andrews (1991) AR(1) automatic plug-in
      Panel:     NW on S_t (cross-sectional sum of violations at
                 each date), scaled by T/N to obtain SE(pi_pooled)
  - p_Kupiec:   Kupiec (1995) LR test, chi2(1)
  - p_cluster:  two-sided z-test with cluster SE =
                 sqrt(var(pihat_i, ddof=1) / J), J = 24 assets
"""

from decimal import Decimal, ROUND_HALF_UP
from scipy.stats import chi2, norm
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.sandwich_covariance import cov_hac

MODEL_ORDER_T6 = ['Chronos-Small', 'Chronos-Mini', 'TimesFM-2.5',
                  'Moirai-2.0', 'Lag-Llama',
                  'GJR-GARCH', 'GARCH-N', 'Hist-Sim', 'EWMA']
ALPHA_T6 = 0.01

MODEL_SUBDIR = {
    'Chronos-Small': 'chronos_small', 'Chronos-Mini': 'chronos_mini',
    'TimesFM-2.5': 'timesfm25', 'Moirai-2.0': 'moirai2',
    'Lag-Llama': 'lagllama',
}
BENCH_SUFFIX = {
    'GJR-GARCH': 'gjr_garch', 'GARCH-N': 'garch_n',
    'Hist-Sim': 'hs', 'EWMA': 'ewma',
}
RET_DIR = DATA_DIR / 'returns'

def parquet_path(model, symbol):
    if model in MODEL_SUBDIR:
        return DATA_DIR / MODEL_SUBDIR[model] / f'{symbol}.parquet'
    return DATA_DIR / 'benchmarks' / f'{symbol}_{BENCH_SUFFIX[model]}.parquet'

# ── Load all_results ─────────────────────────────────────────────
ar = pd.read_csv(RES_DIR / 'all_results.csv')
d01 = ar[ar['alpha'] == ALPHA_T6].copy()

# ── Reconstruct violations & compute statistics ──────────────────
print('Computing panel-pooled backtest (Driscoll-Kraay HAC SE) ...')
rows_t6 = []

for model in MODEL_ORDER_T6:
    sub = d01[d01['model'] == model]
    assets = sorted(sub['symbol'].unique())
    J = len(assets)

    viols = {}
    for sym in assets:
        row = sub[sub['symbol'] == sym].iloc[0]
        pq = pd.read_parquet(parquet_path(model, sym))
        ret = pd.read_csv(RET_DIR / f'{sym}.csv')
        ret['date'] = pd.to_datetime(ret['date'])
        pq.index = pd.to_datetime(pq.index)
        merged = pq[['VaR_0.01']].join(
            ret.set_index('date')['log_return'], how='inner')
        n_cal = int(row['n_cal'])
        n_test = int(row['n_test'])
        test = merged.iloc[n_cal:n_cal + n_test]
        v = (test['log_return'] < (test['VaR_0.01'] - row['qV'])).astype(int)
        viols[sym] = v

    n_panel    = sum(len(v) for v in viols.values())
    total_viol = int(sum(v.sum() for v in viols.values()))
    pi_pooled  = total_viol / n_panel

    # Kupiec LR
    lr = -2 * (total_viol * np.log(ALPHA_T6 / pi_pooled)
               + (n_panel - total_viol) * np.log((1 - ALPHA_T6) / (1 - pi_pooled)))
    p_kupiec = 1 - chi2.cdf(lr, 1)

    # Driscoll-Kraay HAC SE
    all_dates = sorted(set().union(*(v.index for v in viols.values())))
    panel_df = pd.DataFrame(index=all_dates)
    for sym, v in viols.items():
        panel_df[sym] = v
    S_t = panel_df.sum(axis=1).values.astype(float)
    T = len(S_t)
    ols = OLS(S_t, np.ones((T, 1))).fit()
    hac_se = np.sqrt(cov_hac(ols)[0, 0]) * T / n_panel

    # Cluster-robust z and p
    pihat_assets = np.array([
        sub[sub['symbol'] == sym].iloc[0]['pihat_cp'] for sym in assets])
    cluster_se = np.sqrt(np.var(pihat_assets, ddof=1) / J)
    z_cluster = (pi_pooled - ALPHA_T6) / cluster_se
    p_cluster = 2 * (1 - norm.cdf(abs(z_cluster)))

    rows_t6.append({
        'model': model, 'N_panel': n_panel,
        'total_viol': total_viol, 'pi_pooled': pi_pooled,
        'HAC_SE': hac_se, 'p_kupiec': p_kupiec,
        'z_cluster': z_cluster, 'p_cluster': p_cluster,
    })

    print(f'  {model:16s}  N={n_panel}  V={total_viol}  '
          f'pi={pi_pooled:.6f}  HAC_SE={hac_se:.6f}  '
          f'p_kup={p_kupiec:.4f}  p_cl={p_cluster:.4f}')

result_t6 = pd.DataFrame(rows_t6).set_index('model')

# ── Save CSV ─────────────────────────────────────────────────────
result_t6.to_csv(OUT_DIR / 'tab_panel_pooled.csv')

# ── Format helpers (round-half-up) ───────────────────────────────
def _rhu(x, dp):
    d = Decimal(str(x)).quantize(Decimal(10) ** -dp, rounding=ROUND_HALF_UP)
    return format(d, f'.{dp}f')

def fmt_n(n):
    return f'{int(n):,}'.replace(',', '{,}')

def fmt_pi(x):
    return '.' + _rhu(x, 4)[2:]

def fmt_se(x):
    return '.' + _rhu(x, 4)[2:]

def fmt_p(x):
    s3 = _rhu(x, 3)
    if s3 == '0.000':
        return '.' + _rhu(x, 4)[2:]
    return '.' + s3[2:]

# ── Build LaTeX ──────────────────────────────────────────────────
lines = [
    r'\begin{tabular}{@{}lrrrrrr@{}}',
    r'\toprule',
    r'Model & $N_{\text{panel}}$ & Violations',
    r'& Corr.\ $\hat\pi$ & HAC SE',
    r'& $p_{\text{Kup}}$',
    r'& $p_{\text{cluster}}$ \\',
    r'\midrule',
]

for i, model in enumerate(MODEL_ORDER_T6):
    r = result_t6.loc[model]

    label = model
    if model == 'TimesFM-2.5':
        label = 'TimesFM 2.5'
    elif model == 'Moirai-2.0':
        label = 'Moirai 2.0'
    elif model == 'Hist-Sim':
        label = r'Hist.\ Sim.'

    line = (f'{label} & {fmt_n(r["N_panel"])} & {int(r["total_viol"])}'
            f' & {fmt_pi(r["pi_pooled"])}\n'
            f'& {fmt_se(r["HAC_SE"])} & {fmt_p(r["p_kupiec"])}'
            f' & {fmt_p(r["p_cluster"])} \\\\')
    lines.append(line)
    if i == 4:
        lines.append(r'\midrule')

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')

tex = '\n'.join(lines) + '\n'
tex_path = OUT_DIR / 'tab_panel_pooled.tex'
tex_path.write_text(tex)
print(f'\nSaved {tex_path.name}')
print(tex)

## Table 7 — Panel-pooled backtest by asset class (α = 0.01)

In [ ]:
"""
CO_multi_quantile_panel — Panel-pooled backtest by asset class.
Produces tab_panel_by_class.tex  (Table 7 in the paper).

For each of five asset classes (Equity, Bond, Commodity, Crypto, FX),
reports N_assets, N_panel (sum of test observations across all
model-asset pairs), mean corrected coverage rate, and Green-zone rate
under Basel traffic-light classification.
"""

ALPHA_T7 = 0.01

CLASS_MAP = {
    'SP500': 'Equity', 'STOXX': 'Equity', 'GDAXI': 'Equity',
    'FCHI': 'Equity', 'FTSE100': 'Equity', 'ICLN': 'Equity',
    'NIKKEI': 'Equity', 'HSI': 'Equity', 'BOVESPA': 'Equity',
    'NIFTY': 'Equity', 'ASX200': 'Equity',
    'CBU0': 'Bond', 'TLT': 'Bond', 'IBGL': 'Bond',
    'DJCI': 'Commodity', 'GOLD': 'Commodity', 'WTI': 'Commodity',
    'NATGAS': 'Commodity',
    'BTC': 'Crypto', 'ETH': 'Crypto',
    'EURUSD': 'FX', 'GBPUSD': 'FX', 'USDJPY': 'FX', 'AUDUSD': 'FX',
}
CLASS_ORDER = ['Equity', 'Bond', 'Commodity', 'Crypto', 'FX']

# ── Load ───────────────────────────────────────────────────────────
df_t7 = pd.read_csv(RES_DIR / 'all_results.csv')
d01_t7 = df_t7[df_t7['alpha'] == ALPHA_T7].copy()
d01_t7['asset_class'] = d01_t7['symbol'].map(CLASS_MAP)
assert d01_t7['asset_class'].notna().all()

print(f'Loaded {len(d01_t7)} rows at alpha={ALPHA_T7} '
      f'({d01_t7["model"].nunique()} models, '
      f'{d01_t7["symbol"].nunique()} assets)')

# ── Compute per-class statistics ──────────────────────────────────
rows_t7 = []
for cls in CLASS_ORDER:
    sub = d01_t7[d01_t7['asset_class'] == cls]
    n_assets = sub['symbol'].nunique()
    n_panel  = int(sub['n_test'].sum())
    pi_mean  = sub['pihat_cp'].mean()
    green    = int((sub['TL_cp'] == 'Green').sum())
    total    = len(sub)

    rows_t7.append({
        'asset_class': cls, 'n_assets': n_assets,
        'n_panel': n_panel, 'pi_mean': pi_mean,
        'green': green, 'total': total,
    })
    print(f'  {cls:12s}  N={n_assets}  N_panel={n_panel:>9,}  '
          f'pi={pi_mean:.4f}  Green={green}/{total} '
          f'({100*green/total:.0f}%)')

result_t7 = pd.DataFrame(rows_t7)

# ── Save CSV ──────────────────────────────────────────────────────
result_t7.to_csv(OUT_DIR / 'tab_panel_by_class.csv', index=False)

# ── Format helpers ────────────────────────────────────────────────
def fmt_n_t7(n):
    return f'{int(n):,}'.replace(',', '{,}')

def fmt_pi_t7(x):
    d = Decimal(str(x)).quantize(Decimal('0.001'), rounding=ROUND_HALF_UP)
    return '.' + format(d, '.3f')[2:]

# ── Build LaTeX ───────────────────────────────────────────────────
lines = [
    r'\begin{tabular}{@{}lrrrr@{}}',
    r'\toprule',
    r'Asset class & $N$ assets & $N_{\text{panel}}$',
    r'& Corr.\ $\hat\pi$ & Green rate \\',
    r'\midrule',
]

for _, r in result_t7.iterrows():
    pct = int(round(100 * r['green'] / r['total']))
    line = (f'{r["asset_class"]:10s} & {r["n_assets"]:2d}'
            f' & {fmt_n_t7(r["n_panel"])} & {fmt_pi_t7(r["pi_mean"])}\n'
            f'& {r["green"]}/{r["total"]} ({pct}\\%) \\\\')
    lines.append(line)

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')

tex = '\n'.join(lines) + '\n'
tex_path = OUT_DIR / 'tab_panel_by_class.tex'
tex_path.write_text(tex)
print(f'\nSaved {tex_path.name}')
print(tex)